# nano CHSM8

The whole recipe at toy scale, on a laptop CPU: pretraining on games, then SFT with text. Same code as the real
models (`scripts/`); only the sizes in `configs/nano.yaml` and `configs/nano_sft.yaml` are small. Set `LIMIT` to
train on more games.

In [ ]:
LIMIT = 3000        # games for pretraining; SFT uses LIMIT // 10 described sides per split
import os, subprocess, sys
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, ".")
run = lambda *cmd: subprocess.run([sys.executable, *cmd], check=True)

## 1. Data

One file of one month of chsm8-pre-training (150 MB) is enough for the pretraining step.

In [ ]:
from huggingface_hub import hf_hub_download
hf_hub_download("LegumMagister/chsm8-pre-training", "data/2026-06/part-00000.parquet", repo_type="dataset",
                local_dir="data/nano_src")
run("scripts/data/prepare_pretraining.py", "--out", "data/nano", "--months", "2026-06", "--max-games", str(LIMIT),
    "--source", "data/nano_src", "--workers", "1")

## 2. Pretraining

The games-only model: 4 layers x 128, about 1M parameters. Loss is per field (src, dst, piece, promo) plus a penalty
on probability given to illegal squares and pieces.

In [ ]:
run("scripts/train/pretrain.py", "configs/nano.yaml")

## 3. SFT with text

Needs the text encoder, EmbeddingGemma-300m: accept its license on Hugging Face and log in (`huggingface-cli login`).
The described sides come from chsm8-post-training (validation and test splits are small; the first train file is
about 450 MB).

In [ ]:
for f in ["data/train/part-00000.parquet", "data/validation/part-00000.parquet", "data/test/part-00000.parquet"]:
    hf_hub_download("LegumMagister/chsm8-post-training", f, repo_type="dataset", local_dir="data/nano_post")
run("scripts/data/prepare_posttraining.py", "--sides", "data/nano_sides", "--post", "data/nano_post",
    "--max-sides", str(LIMIT // 10), "--workers", "4")
run("scripts/train/sft.py", "configs/nano_sft.yaml")

## 4. Play

A nano model plays poorly, but legally: every move is chosen among the legal ones.

In [ ]:
from chsm8.play import Player
from chsm8.text import TextEncoder
from chsm8 import checkpoint
model = checkpoint.load("runs/nano_sft/model.pt", device="cpu")
p = Player(model, TextEncoder(device="cpu"))
for m in p.scores("1. e4 e5 2. Nf3", "attack the king at any cost")[:5]:
    print(f"{m['san']:6s} {m['logp']:.2f}")